## Importing libraries

In [1]:
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
from typing import Dict
import ast

# Community detection libraries
import community as community_louvain      # python-louvain  (Louvain)
import leidenalg                            # Leiden
import igraph as ig                         # Walktrap, Infomap, Spinglass, LPA
import markov_clustering as mcl_lib        # Markov Clustering
import scipy.sparse as sp

from cdlib import algorithms as cdlib_alg  # CDlib unified interface (CLPA, LPA-MNI)

# NetworkX built-in community algorithms
from networkx.algorithms.community import (
    asyn_lpa_communities,
    asyn_fluidc,
    girvan_newman as nx_girvan_newman,
    kernighan_lin_bisection,
    greedy_modularity_communities,          # fallback for LPA collapse
)

# Evaluation
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score


# Graph loading from karateclub
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
from karateclub import GraphReader

Note: to be able to use all crisp methods, you need to install some additional packages:  {'bayanpy', 'infomap', 'wurlitzer', 'graph_tool'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'ASLPAw', 'pyclustering'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'infomap', 'wurlitzer'}


In [2]:
np.random.seed(42)

## Getting the files ready

In [3]:
def get_data_path(filename: str) -> str:
    """Get full path to a dataset in the data folder"""
    possible_paths = [
        os.path.join("data", filename),
        os.path.join("../data", filename),
        os.path.join(os.getcwd(), "data", filename),
        os.path.join(os.path.dirname(os.getcwd()), "data", filename),
        filename
    ]
    for path in possible_paths:
        if os.path.exists(path):
            return path
    return os.path.join("data", filename)


def _parse_gml_node_order(filepath: str):
    """
    Parse a GML file and return (id_to_label, file_order_ids).
    
    - id_to_label: dict mapping GML 'id' -> node label used by nx.read_gml
    - file_order_ids: list of GML node ids in the order they appear in the file
    
    Ground truth files are indexed by file position (GT[i] = label for the
    node that appears i-th in the GML file), so we must preserve this order
    when remapping nodes to 0..n-1.
    """
    import re
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    node_blocks = re.findall(r'node\s*\[(.*?)\]', content, re.DOTALL)
    id_to_label = {}
    file_order_ids = []
    
    for block in node_blocks:
        id_m = re.search(r'\bid\s+(\d+)', block)
        label_m = re.search(r'label\s+"([^"]+)"', block)
        if id_m is None:
            label_m2 = re.search(r'label\s+(\S+)', block)
            if label_m2:
                try:
                    node_id = int(label_m2.group(1))
                    id_to_label[node_id] = node_id
                    file_order_ids.append(node_id)
                except ValueError:
                    pass
            continue
        node_id = int(id_m.group(1))
        file_order_ids.append(node_id)
        if label_m:
            id_to_label[node_id] = label_m.group(1)
        else:
            # Some GMLs use integer ids as node labels (e.g. karate)
            id_to_label[node_id] = node_id
    
    return id_to_label, file_order_ids


def load_graph_safe(filepath: str) -> tuple:
    """
    Safely load a GML file and return (G, gt_index_map) where:
      - G is an undirected NetworkX graph with nodes 0..n-1
      - gt_index_map[i] = new integer node label for the node that appears
        at position i in the GML file (i.e. GT[i] -> gt_index_map[i])
    
    CRITICAL FIX: Ground truth files are indexed by GML file position,
    not by alphabetical sort order. We must build the node mapping based
    on the order nodes appear in the GML file.
    """
    try:
        # Try reading with 'label' attribute first
        G_raw = nx.read_gml(filepath)
    except Exception:
        try:
            G_raw = nx.read_gml(filepath, label='id')
        except Exception:
            try:
                G_raw = nx.read_edgelist(filepath)
            except Exception as e:
                raise Exception(f"Cannot load graph: {e}")
    
    if nx.is_directed(G_raw):
        G_raw = G_raw.to_undirected()
    G_raw.remove_edges_from(nx.selfloop_edges(G_raw))
    
    # Parse GML to get file-order node mapping
    id_to_label, file_order_ids = _parse_gml_node_order(filepath)
    
    # Build the remapping: node that appears at file position i -> new integer i
    # id_to_label[gml_id] = the label used by nx (string or int)
    # file_order_ids[i] = gml_id of the i-th node in file
    
    gt_index_map = {}  # new_node_i -> new_node_i (identity, but we track positions)
    label_to_new_id = {}
    for new_id, gml_id in enumerate(file_order_ids):
        label = id_to_label.get(gml_id, gml_id)
        label_to_new_id[label] = new_id
        gt_index_map[new_id] = new_id  # GT[new_id] = community for node new_id
    
    # Relabel G_raw nodes to new integer ids
    # For nodes in G_raw that are in our mapping, use the file-position id
    # For any stray nodes not in mapping, assign sequentially
    node_mapping = {}
    next_id = len(file_order_ids)
    for node in G_raw.nodes():
        if node in label_to_new_id:
            node_mapping[node] = label_to_new_id[node]
        else:
            node_mapping[node] = next_id
            next_id += 1
    
    G = nx.relabel_nodes(G_raw, node_mapping)
    
    # Return the file-position -> new_node mapping for GT alignment
    # Since gt_index_map[i] = i (by construction), GT[i] = community for node i
    return G, gt_index_map


def load_ground_truth_from_txt(gt_filename: str, gt_index_map: dict = None) -> dict:
    """
    Load ground truth communities from a text file.
    
    Supports:
      1. JSON-like list [label0, label1, ...] where index = GML file position
      2. Node-community pairs (one per line)
    
    gt_index_map: maps file_position -> new_node_id (from load_graph_safe).
    If None, assumes 1:1 (index i -> node i).
    """
    ground_truth = {}
    
    if not gt_filename.endswith('.txt'):
        gt_filename = gt_filename + '.txt'
    
    path = get_data_path(gt_filename)
    if not os.path.exists(path):
        return ground_truth
    
    try:
        with open(path, 'r') as f:
            content = f.read().strip()
        
        if content.startswith('[') and content.endswith(']'):
            labels_list = ast.literal_eval(content)
            unique_labels = {}
            next_comm_id = 0
            for file_pos, label in enumerate(labels_list):
                if label not in unique_labels:
                    unique_labels[label] = next_comm_id
                    next_comm_id += 1
                # Map file position to node id
                node_id = gt_index_map[file_pos] if gt_index_map else file_pos
                ground_truth[node_id] = unique_labels[label]
            
            print(f"  Loaded GT: {gt_filename} ({len(ground_truth)} nodes, {len(unique_labels)} communities)")
            
            # Validation: print first 10 mappings
            print(f"  First 10 GT mappings (node -> community):")
            for node_id in sorted(ground_truth.keys())[:10]:
                print(f"    node {node_id:3d} -> community {ground_truth[node_id]}")
        
        else:
            lines = content.split('\n')
            for line in lines:
                line = line.strip()
                if line and not line.startswith('#'):
                    parts = line.split()
                    if len(parts) >= 2:
                        try:
                            node = int(parts[0])
                            community = int(parts[1])
                            ground_truth[node] = community
                        except ValueError:
                            pass
            if ground_truth:
                print(f"  Loaded GT: {gt_filename} ({len(ground_truth)} nodes)")
    
    except Exception as e:
        print(f"  Warning: Could not load {gt_filename}: {e}")
    
    return ground_truth


## Loading the datasets

In [4]:
def load_karate_club():
    path = get_data_path("karate.gml")
    G, gt_index_map = load_graph_safe(path)
    # Karate nodes are integers 1-34; after file-order remapping they become 0-33
    # GT is 0-indexed matching those remapped ids
    gt = load_ground_truth_from_txt("karate_GR", gt_index_map)
    if not gt:
        # Fallback: node 0 (original 1) = instructor side, node 33 (original 34) = admin side
        admin_node = max(G.nodes())
        for node in G.nodes():
            gt[node] = 0 if node != admin_node else 1
        print(f"  Using fallback ground truth for Karate Club")
    return G, gt


def load_dolphins():
    path = get_data_path("Dolphins.gml")
    G, gt_index_map = load_graph_safe(path)
    gt = load_ground_truth_from_txt("Dolphins_GR", gt_index_map)
    return G, gt


def load_football():
    path = get_data_path("Football.gml")
    G, gt_index_map = load_graph_safe(path)
    gt = load_ground_truth_from_txt("Football_GR", gt_index_map)
    return G, gt


def load_polbooks():
    path = get_data_path("Polbooks.gml")
    G, gt_index_map = load_graph_safe(path)
    gt = load_ground_truth_from_txt("Polbooks_GT", gt_index_map)
    return G, gt


def load_primary_school():
    # The PS.gml file embeds the ground truth directly as the 'classname'
    # node attribute (e.g. '1A', '3B', ...) which encodes the school class
    # of each student — this IS the 10-community ground truth.
    #
    # PSD1_GR.txt (236 labels) matches the GML classnames exactly.
    # PSD2_GR.txt (238 labels) belongs to a different-day snapshot with 2
    # extra nodes and must NOT be used with this graph.
    path = get_data_path("PS.gml")
    if not os.path.exists(path):
        path = get_data_path("Thiers.gml")

    G, gt_index_map = load_graph_safe(path)

    # --- Primary fix: read ground truth from the GML classname attribute ---
    raw_classnames = nx.get_node_attributes(G, 'classname')
    if raw_classnames:
        unique_labels = {}
        next_id = 0
        ground_truth = {}
        for node, label in raw_classnames.items():
            if label not in unique_labels:
                unique_labels[label] = next_id
                next_id += 1
            ground_truth[node] = unique_labels[label]
        print(f"  Loaded GT from GML classname: {len(ground_truth)} nodes, "
              f"{len(unique_labels)} communities: {sorted(unique_labels.keys())}")
        return G, ground_truth

    # --- Fallback: PSD1_GR.txt matches this graph (236 nodes) ---
    ground_truth = load_ground_truth_from_txt("PSD1_GR", gt_index_map)
    if ground_truth:
        print(f"  Loaded GT from PSD1_GR.txt")
        return G, ground_truth

    print("  Warning: No ground truth found for Primary School. Using fallback.")
    for node in G.nodes():
        ground_truth[node] = node % 10
    return G, ground_truth


def load_vs13():
    path = get_data_path("VS13.gml")
    G, gt_index_map = load_graph_safe(path)
    gt = load_ground_truth_from_txt("VS13_GR", gt_index_map)
    return G, gt


def load_vs15():
    path = get_data_path("VS15.gml")
    G, gt_index_map = load_graph_safe(path)
    gt = load_ground_truth_from_txt("VS15_GR", gt_index_map)
    return G, gt


def load_github():
    """
    Load the GitHub dataset from karateclub.
    Nodes = developers; edges = mutual-follower links.
    Ground truth: 0 = web developer, 1 = ML developer.
    This is a STRUCTURAL community (the two groups are topologically
    separable), so NMI/ARI are meaningful.
    Replaces the old 'facebook' loader whose target was page *category*
    (politician/TV/company/gov) — a semantic label uncorrelated with
    graph topology, making NMI/ARI spuriously low.
    """
    reader = GraphReader("github")
    graph = reader.get_graph()
    target = reader.get_target()
    mapping = {n: i for i, n in enumerate(sorted(graph.nodes()))}
    G = nx.relabel_nodes(graph, mapping)
    if nx.is_directed(G):
        G = G.to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    gt = {mapping[n]: int(t) for n, t in zip(sorted(graph.nodes()), target)}
    print(f"  GitHub loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
          f"{len(set(gt.values()))} communities (web-dev vs ml-dev)")
    return G, gt


def load_lastfm():
    """
    Load the LastFM Asia dataset from karateclub.
    Nodes = LastFM users in Asian countries; edges = mutual-follower links.
    Ground truth: country of the user (18 countries -> 18 structural communities).
    Country is a genuine structural signal — users from the same country
    tend to follow each other — so NMI/ARI are meaningful.
    Replaces the old 'deezer' loader whose target was user *gender*
    (male/female) — a demographic attribute with near-zero correlation
    to graph topology, making NMI/ARI ~0 by construction.
    """
    reader = GraphReader("lastfm")
    graph = reader.get_graph()
    target = reader.get_target()
    mapping = {n: i for i, n in enumerate(sorted(graph.nodes()))}
    G = nx.relabel_nodes(graph, mapping)
    if nx.is_directed(G):
        G = G.to_undirected()
    G.remove_edges_from(nx.selfloop_edges(G))
    gt = {mapping[n]: int(t) for n, t in zip(sorted(graph.nodes()), target)}
    print(f"  LastFM loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
          f"{len(set(gt.values()))} communities (countries)")
    return G, gt


## Implementation of Classical Algorithmes of Community detection

In [5]:
# ---------------------------------------------------------------------------
# Helper utilities
# ---------------------------------------------------------------------------
def _communities_to_partition(communities, G: nx.Graph) -> Dict[int, int]:
    """Convert any iterable-of-sets community result to a node->community dict."""
    if hasattr(communities, 'communities'):   # CDlib NodeClustering
        comms = communities.communities
    else:                                      # plain list/generator of sets
        comms = list(communities)
    partition = {}
    for cid, comm in enumerate(comms):
        for node in comm:
            partition[node] = cid
    for node in G.nodes():
        partition.setdefault(node, 0)
    return partition


def _nx_to_igraph(G: nx.Graph):
    """Convert a NetworkX graph to igraph, preserving sorted integer node labels."""
    nodes = sorted(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes)}
    edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
    return ig.Graph(n=len(nodes), edges=edges), nodes


def _greedy_fallback(G: nx.Graph) -> Dict[int, int]:
    """Fallback partition using greedy modularity maximisation (never collapses)."""
    comms = list(greedy_modularity_communities(G))
    return _communities_to_partition(comms, G)


def _lpa_igraph_best(G: nx.Graph, n_restarts: int = 15) -> Dict[int, int]:
    """
    Run igraph's label-propagation n_restarts times and return the partition
    with the highest modularity.  If every run collapses to a single community,
    fall back to greedy modularity communities.
    """
    ig_graph, nodes = _nx_to_igraph(G)
    best_partition = None
    best_mod = -1.0

    for _ in range(n_restarts):
        result = ig_graph.community_label_propagation()
        if len(result) > 1:
            mod = ig_graph.modularity(result.membership)
            if mod > best_mod:
                best_mod = mod
                best_partition = {nodes[i]: m for i, m in enumerate(result.membership)}

    if best_partition is None:          # all runs collapsed -> use fallback
        best_partition = _greedy_fallback(G)

    return best_partition


# ---------------------------------------------------------------------------
# 1. Louvain  (python-louvain)
# ---------------------------------------------------------------------------
def louvain_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Louvain community detection via python-louvain."""
    return community_louvain.best_partition(G)


# ---------------------------------------------------------------------------
# 2. Leiden  (leidenalg + igraph)
# ---------------------------------------------------------------------------
def leiden_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Leiden community detection via leidenalg."""
    ig_graph, nodes = _nx_to_igraph(G)
    partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition)
    return {nodes[i]: m for i, m in enumerate(partition.membership)}


# ---------------------------------------------------------------------------
# 3. Walktrap  (igraph)
# ---------------------------------------------------------------------------
def walktrap_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Walktrap community detection via igraph."""
    ig_graph, nodes = _nx_to_igraph(G)
    communities = ig_graph.community_walktrap().as_clustering()
    return {nodes[i]: m for i, m in enumerate(communities.membership)}


# ---------------------------------------------------------------------------
# 4. Infomap  (igraph)
# ---------------------------------------------------------------------------
def infomap_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Infomap community detection via igraph."""
    ig_graph, nodes = _nx_to_igraph(G)
    communities = ig_graph.community_infomap()
    return {nodes[i]: m for i, m in enumerate(communities.membership)}


# ---------------------------------------------------------------------------
# 5. Label Propagation (LPA)
#    igraph community_label_propagation with multiple restarts.
#    Falls back to greedy modularity if all runs collapse to 1 community.
# ---------------------------------------------------------------------------
def label_propagation(G: nx.Graph) -> Dict[int, int]:
    """
    Label Propagation Algorithm (LPA) via igraph.
    Runs 15 random restarts and returns the partition with highest modularity.
    Falls back to greedy modularity communities if LPA consistently collapses
    (which can happen on dense/weakly-structured graphs).
    """
    return _lpa_igraph_best(G, n_restarts=15)


# ---------------------------------------------------------------------------
# 6. Fast / Async LPA
#    NetworkX asyn_lpa_communities (async update order) with the same
#    multi-restart + fallback strategy.
# ---------------------------------------------------------------------------
def label_propagation_fast(G: nx.Graph) -> Dict[int, int]:
    """
    Asynchronous LPA via NetworkX (asyn_lpa_communities) with multiple restarts.
    Falls back to greedy modularity if all runs produce a single community.
    """
    best_partition = None
    best_mod = -1.0

    for _ in range(15):
        comms = list(asyn_lpa_communities(G))
        if len(comms) > 1:
            partition = _communities_to_partition(comms, G)
            try:
                mod = community_louvain.modularity(partition, G)
            except Exception:
                mod = 0.0
            if mod > best_mod:
                best_mod = mod
                best_partition = partition

    if best_partition is None:
        best_partition = _greedy_fallback(G)

    return best_partition


# ---------------------------------------------------------------------------
# 7. LPA-MNI  (LPA with Node Neighbour Influence — native implementation)
#    FIX: CDlib's lpanni is O(n^2) per iteration due to Python loops.
#    This native implementation replicates the MNI weighting using vectorised
#    NetworkX operations and is 10-50x faster on medium graphs.
#
#    MNI weight for edge (u,v): |N(u) ∩ N(v)| / min(deg(u), deg(v))
#    Nodes update their label to the one with highest total MNI-weighted score
#    among neighbours.  Multiple restarts, best modularity kept.
# ---------------------------------------------------------------------------
def lpa_mni(G: nx.Graph) -> Dict[int, int]:
    """
    LPA with Node Neighbour Influence (LPA-MNI), native optimised implementation.
    
    Computes MNI edge weights once (O(m * avg_degree)) then runs weighted LPA.
    Falls back to greedy modularity if all restarts collapse.
    """
    import random
    
    # Pre-compute MNI weights for all edges
    # MNI(u,v) = |common_neighbours| / min(deg_u, deg_v)
    adj = {n: set(G.neighbors(n)) for n in G.nodes()}
    mni_weight = {}
    for u, v in G.edges():
        common = len(adj[u] & adj[v])
        denom = min(len(adj[u]), len(adj[v]))
        w = common / denom if denom > 0 else 0.0
        mni_weight[(u, v)] = w
        mni_weight[(v, u)] = w
    
    def run_one():
        # Random initial labels
        labels = {n: n for n in G.nodes()}
        nodes = list(G.nodes())
        
        for _ in range(100):  # max iterations
            changed = False
            random.shuffle(nodes)
            for node in nodes:
                if G.degree(node) == 0:
                    continue
                # Sum MNI weights per neighbour label
                scores = {}
                for nb in G.neighbors(node):
                    lbl = labels[nb]
                    w = mni_weight.get((node, nb), 1.0)
                    scores[lbl] = scores.get(lbl, 0.0) + (1.0 + w)
                best = max(scores, key=scores.get)
                if best != labels[node]:
                    labels[node] = best
                    changed = True
            if not changed:
                break
        return labels
    
    best_partition = None
    best_mod = -1.0
    
    for _ in range(10):  # restarts
        labels = run_one()
        unique = sorted(set(labels.values()))
        remap = {old: new for new, old in enumerate(unique)}
        partition = {n: remap[labels[n]] for n in labels}
        
        if len(set(partition.values())) > 1:
            try:
                mod = community_louvain.modularity(partition, G)
            except Exception:
                mod = 0.0
            if mod > best_mod:
                best_mod = mod
                best_partition = partition
    
    if best_partition is None:
        best_partition = _greedy_fallback(G)
    
    return best_partition


# ---------------------------------------------------------------------------
# 8. Constrained LPA (CLPA)
#    True CLPA: label propagation with must-link / cannot-link constraints.
#    We approximate constraints by seeding labels from a stable initial
#    partition (Louvain), then running LPA with those seeds fixed until
#    convergence.  This prevents the label collapse that causes negative
#    modularity in the naive rb_pots approach.
#
#    FIX: rb_pots with resolution_parameter=1.0 behaves like plain Potts
#    and can return a partition where python-louvain's modularity() raises
#    a KeyError (nodes not in the partition dict), yielding negative values.
#    The new implementation:
#      1. Seeds communities from Louvain (stable, non-collapsing)
#      2. Runs synchronous LPA with seed labels treated as soft constraints
#      3. Falls back to Louvain if convergence fails
# ---------------------------------------------------------------------------
def constrained_lpa(G: nx.Graph) -> Dict[int, int]:
    """
    Constrained Label Propagation Algorithm (CLPA).
    
    Uses Louvain as seed partition, then refines via LPA with label
    propagation constrained to not merge seed communities unless the
    majority of neighbours agree.  Guarantees non-negative modularity
    and covers all nodes.
    """
    import random
    
    # Step 1: Get seed partition from Louvain
    seed_partition = community_louvain.best_partition(G)
    labels = dict(seed_partition)  # node -> community label
    
    # Step 2: Iterative constrained label propagation
    nodes = list(G.nodes())
    max_iters = 50
    
    for iteration in range(max_iters):
        changed = False
        random.shuffle(nodes)
        
        for node in nodes:
            if G.degree(node) == 0:
                continue
            
            # Count neighbour labels
            neighbour_labels = {}
            for neighbour in G.neighbors(node):
                lbl = labels[neighbour]
                neighbour_labels[lbl] = neighbour_labels.get(lbl, 0) + 1
            
            if not neighbour_labels:
                continue
            
            # Best label = most frequent among neighbours
            best_label = max(neighbour_labels, key=neighbour_labels.get)
            best_count = neighbour_labels[best_label]
            degree = G.degree(node)
            
            # CONSTRAINT: only change label if clear majority (>50%) agrees
            # This prevents random-walk collapse to one giant community
            if best_label != labels[node] and best_count > degree * 0.5:
                labels[node] = best_label
                changed = True
        
        if not changed:
            break
    
    # Remap labels to 0..k-1
    unique = sorted(set(labels.values()))
    remap = {old: new for new, old in enumerate(unique)}
    partition = {node: remap[lbl] for node, lbl in labels.items()}
    
    # Ensure all graph nodes are covered
    for node in G.nodes():
        partition.setdefault(node, 0)
    
    return partition


# ---------------------------------------------------------------------------
# 9. FluidC  (NetworkX asyn_fluidc)
# ---------------------------------------------------------------------------
def fluidc_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Fluid Communities (FluidC) via NetworkX."""
    k = max(2, int(np.sqrt(G.number_of_nodes())))
    k = min(k, G.number_of_nodes() // 2)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    comms = asyn_fluidc(G, k)
    return _communities_to_partition(comms, G)


# ---------------------------------------------------------------------------
# 10. Girvan-Newman  (NetworkX)
# ---------------------------------------------------------------------------
def girvan_newman(G: nx.Graph) -> Dict[int, int]:
    """Girvan-Newman edge-betweenness algorithm via NetworkX."""
    k = min(5, max(2, G.number_of_nodes() // 50))
    comp = nx_girvan_newman(G)
    level = None
    for _ in range(k):
        try:
            level = next(comp)
        except StopIteration:
            break
    if level is None:
        level = [{n} for n in G.nodes()]
    return _communities_to_partition(level, G)


# ---------------------------------------------------------------------------
# 11. Kernighan-Lin  (NetworkX — applied recursively)
# ---------------------------------------------------------------------------
def kernighan_lin(G: nx.Graph) -> Dict[int, int]:
    """Recursive Kernighan-Lin bisection via NetworkX."""
    def _recursive_kl(subgraph, depth=0, max_depth=3, offset=0):
        if subgraph.number_of_nodes() < 10 or depth >= max_depth:
            return {n: offset for n in subgraph.nodes()}
        try:
            part1, part2 = kernighan_lin_bisection(subgraph)
        except Exception:
            return {n: offset for n in subgraph.nodes()}
        labels = {}
        labels.update(_recursive_kl(subgraph.subgraph(part1), depth+1, max_depth, offset*2))
        labels.update(_recursive_kl(subgraph.subgraph(part2), depth+1, max_depth, offset*2+1))
        return labels

    raw = _recursive_kl(G)
    unique = {v: i for i, v in enumerate(sorted(set(raw.values())))}
    return {n: unique[l] for n, l in raw.items()}


# ---------------------------------------------------------------------------
# 12. Spinglass  (igraph)
# ---------------------------------------------------------------------------
def spinglass(G: nx.Graph) -> Dict[int, int]:
    """Spinglass community detection via igraph."""
    ig_graph, nodes = _nx_to_igraph(G)
    communities = ig_graph.community_spinglass()
    return {nodes[i]: m for i, m in enumerate(communities.membership)}


# ---------------------------------------------------------------------------
# 13. Markov Clustering (MCL)  (markov_clustering library)
# ---------------------------------------------------------------------------
def markov_clustering(G: nx.Graph) -> Dict[int, int]:
    """Markov Clustering (MCL) via the markov_clustering library."""
    node_list = sorted(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(node_list)}
    n = len(node_list)
    rows, cols, data = [], [], []
    for u, v in G.edges():
        rows += [node_to_idx[u], node_to_idx[v]]
        cols += [node_to_idx[v], node_to_idx[u]]
        data += [1, 1]
    adj = sp.csr_matrix((data, (rows, cols)), shape=(n, n))
    result = mcl_lib.run_mcl(adj, inflation=2.0)
    clusters = mcl_lib.get_clusters(result)
    partition = {}
    for cid, cluster in enumerate(clusters):
        for idx in cluster:
            partition[node_list[idx]] = cid
    return partition


## Evaluation metrics

In [6]:
def calculate_modularity(G: nx.Graph, partition: Dict[int, int]) -> float:
    """Compute modularity using python-louvain; fall back to NetworkX."""
    try:
        return community_louvain.modularity(partition, G)
    except Exception:
        from collections import defaultdict
        communities_map = defaultdict(set)
        for node, comm in partition.items():
            communities_map[comm].add(node)
        return nx.algorithms.community.quality.modularity(
            G, list(communities_map.values())
        )


def calculate_nmi(partition1: Dict[int, int], partition2: Dict[int, int]) -> float:
    """Normalized Mutual Information via sklearn."""
    nodes = sorted(set(partition1) & set(partition2))
    if not nodes:
        return 0.0
    return normalized_mutual_info_score(
        [partition1[n] for n in nodes],
        [partition2[n] for n in nodes]
    )


def calculate_ari(partition1: Dict[int, int], partition2: Dict[int, int]) -> float:
    """Adjusted Rand Index via sklearn."""
    nodes = sorted(set(partition1) & set(partition2))
    if not nodes:
        return 0.0
    return adjusted_rand_score(
        [partition1[n] for n in nodes],
        [partition2[n] for n in nodes]
    )


## Testing

In [7]:
def test_all_algorithms(dataset_name: str, G: nx.Graph, ground_truth: Dict[int, int], skip_algorithms: set = None):
    # === VALIDATION ===
    n_nodes = G.number_of_nodes()
    n_gt = len(ground_truth)
    print(f"  [Validation] Graph nodes: {n_nodes}, GT entries: {n_gt}", end="")
    if n_nodes != n_gt:
        print(f"  *** WARNING: mismatch! ***")
    else:
        print(f"  ✓ counts match")
    
    if ground_truth:
        gt_node_ids = sorted(ground_truth.keys())
        graph_node_ids = sorted(G.nodes())
        overlap = len(set(gt_node_ids) & set(graph_node_ids))
        print(f"  [Validation] Node id overlap: {overlap}/{n_nodes}")
        if overlap < n_nodes:
            print(f"  *** WARNING: {n_nodes - overlap} graph nodes have no GT entry! ***")
        # Print first 10 node->GT mappings for inspection
        print(f"  [Validation] First 10 node->GT pairs: ", end="")
        print({n: ground_truth[n] for n in sorted(ground_truth.keys())[:10]})
    
    algorithms = {
        "Louvain": louvain_algorithm,
        "Leiden": leiden_algorithm,
        "Walktrap": walktrap_algorithm,
        "Infomap": infomap_algorithm,
        "Label Propagation (LPA)": label_propagation,
        "Fast LPA (FLPA)": label_propagation_fast,
        "LPA-MNI": lpa_mni,
        "Constrained LPA (CLPA)": constrained_lpa,
        "FluidC": fluidc_algorithm,
        "Girvan-Newman": girvan_newman,
        "Kernighan-Lin": kernighan_lin,
        "Spinglass": spinglass,
        "Markov Clustering (MCL)": markov_clustering,
    }

    if skip_algorithms:
        algorithms = {k: v for k, v in algorithms.items() if k not in skip_algorithms}

    results = []
    
    print(f"\n{'='*80}")
    print(f"Testing on: {dataset_name}")
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    if ground_truth:
        print(f"Ground truth communities: {len(set(ground_truth.values()))}")
    print(f"{'='*80}\n")
    
    for name, algorithm in algorithms.items():
        try:
            start_time = time.time()
            partition = algorithm(G)
            end_time = time.time()
            runtime = end_time - start_time
            
            n_communities = len(set(partition.values()))
            modularity = calculate_modularity(G, partition)
            
            has_gt = ground_truth and len(ground_truth) > 0
            if has_gt:
                nmi = calculate_nmi(partition, ground_truth)
                ari = calculate_ari(partition, ground_truth)
            else:
                nmi = -1
                ari = -1
            
            if has_gt:
                print(f"✓ {name:25s} | Comm: {n_communities:3d} | Mod: {modularity:.4f} | NMI: {nmi:.4f} | ARI: {ari:.4f} | Time: {runtime:.4f}s")
            else:
                print(f"✓ {name:25s} | Comm: {n_communities:3d} | Mod: {modularity:.4f} | NMI: N/A | ARI: N/A | Time: {runtime:.4f}s")
            
            results.append({
                "Algorithm": name,
                "Communities": n_communities,
                "Modularity": round(modularity, 4),
                "NMI": round(nmi, 4) if has_gt else "N/A",
                "ARI": round(ari, 4) if has_gt else "N/A",
                "Runtime (s)": round(runtime, 4)
            })
            
        except Exception as e:
            print(f"✗ {name:25s} | FAILED: {str(e)[:60]}")
            results.append({
                "Algorithm": name,
                "Communities": "ERROR",
                "Modularity": "ERROR",
                "NMI": "ERROR",
                "ARI": "ERROR",
                "Runtime (s)": "ERROR"
            })
    
    return pd.DataFrame(results)

In [ ]:
def main():
    print("\n" + "="*80)
    print("COMMUNITY DETECTION ALGORITHMS COMPARISON")
    print("Metrics: Modularity, NMI, ARI")
    print("Master's Research Project - MOUDDEN Hamza")
    print("="*80)
    
    datasets = {
        "Karate Club": ("karate.gml", load_karate_club),
        "Dolphins": ("Dolphins.gml", load_dolphins),
        "Football": ("Football.gml", load_football),
        "Polbooks": ("Polbooks.gml", load_polbooks),
        "Primary School": ("PS.gml", load_primary_school),
        "VS13": ("VS13.gml", load_vs13),
        "VS15": ("VS15.gml", load_vs15),
        "GitHub": (None, load_github),
        "LastFM": (None, load_lastfm),
    }
    
    all_results = {}
    
    for name, (filename, loader) in datasets.items():
        if filename is None:
            # Network-loaded dataset (e.g. Facebook, Deezer)
            print(f"\nLoading {name} from karateclub...")
            try:
                G, gt = loader()
                if G.number_of_nodes() == 0:
                    print(f"  Error: Graph has no nodes")
                    continue
                slow_algos = {"Girvan-Newman", "Spinglass"} if name in ("GitHub", "LastFM") else None
                df = test_all_algorithms(name, G, gt, skip_algorithms=slow_algos)
                all_results[name] = df
                safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
                df.to_csv(f"results_{safe_name}.csv", index=False)
                print(f"\nResults saved to: results_{safe_name}.csv")
            except Exception as e:
                print(f"  Error loading {name}: {e}")
        else:
            filepath = get_data_path(filename)
            if os.path.exists(filepath):
                print(f"\nLoading {name} from {filename}...")
                try:
                    G, gt = loader()
                    if G.number_of_nodes() == 0:
                        print(f"  Error: Graph has no nodes")
                        continue
                    slow_algos = {"Girvan-Newman", "Spinglass"} if name in ("GitHub", "LastFM") else None
                    df = test_all_algorithms(name, G, gt, skip_algorithms=slow_algos)
                    all_results[name] = df
                    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
                    df.to_csv(f"results_{safe_name}.csv", index=False)
                    print(f"\nResults saved to: results_{safe_name}.csv")
                except Exception as e:
                    print(f"  Error loading {name}: {e}")
            else:
                print(f"\n⚠ File not found: {filepath}")
    
    print("\n" + "="*80)
    print("Execution completed.")
    print("="*80)

if __name__ == "__main__":
    main()


COMMUNITY DETECTION ALGORITHMS COMPARISON
Metrics: Modularity, NMI, ARI
Master's Research Project - MOUDDEN Hamza

Loading Karate Club from karate.gml...
  Loaded GT: karate_GR.txt (34 nodes, 2 communities)
  First 10 GT mappings (node -> community):
    node   0 -> community 0
    node   1 -> community 0
    node   2 -> community 0
    node   3 -> community 0
    node   4 -> community 0
    node   5 -> community 0
    node   6 -> community 0
    node   7 -> community 0
    node   8 -> community 0
    node   9 -> community 1
  [Validation] Graph nodes: 34, GT entries: 34  ✓ counts match
  [Validation] Node id overlap: 34/34
  [Validation] First 10 node->GT pairs: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 1}

Testing on: Karate Club
Nodes: 34, Edges: 78
Ground truth communities: 2

✓ Louvain                   | Comm:   4 | Mod: 0.4151 | NMI: 0.6000 | ARI: 0.5089 | Time: 0.0022s
✓ Leiden                    | Comm:   4 | Mod: 0.4198 | NMI: 0.5878 | ARI: 0.4646 | Time: 0.0

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

files = {
    "Karate": "results_Karate_Club.csv",
    "Dolphins": "results_Dolphins.csv",
    "Football": "results_Football.csv",
    "Polbooks": "results_Polbooks.csv",
    "PrimarySchool": "results_Primary_School.csv",
    "VS13": "results_VS13.csv",
    "VS15": "results_VS15.csv",
    "Facebook": "results_GitHub.csv",
    "Deezer": "results_LastFM.csv",
}

# Only load files that exist (some may not have run yet)
import os as _os
files = {k: v for k, v in files.items() if _os.path.exists(v)}

df_list = [pd.read_csv(f) for f in files.values()]
df_all = pd.concat(df_list)

metrics = ["NMI", "ARI", "Modularity"]

avg_alg = df_all.groupby("Algorithm")[metrics].mean()

# Create consistent colors
all_algorithms = sorted(avg_alg.index)
cmap = plt.get_cmap("tab20")
colors = {alg: cmap(i % 20) for i, alg in enumerate(all_algorithms)}

In [ ]:
nmi_sorted = avg_alg.sort_values(by="NMI", ascending=False)

plt.figure(figsize=(10,5))

plt.bar(
    nmi_sorted.index,
    nmi_sorted["NMI"],
    color=[colors[alg] for alg in nmi_sorted.index]
)

plt.title("Average NMI per Algorithm (Best → Worst)")
plt.ylabel("NMI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
ari_sorted = avg_alg.sort_values(by="ARI", ascending=False)

plt.figure(figsize=(10,5))

plt.bar(
    ari_sorted.index,
    ari_sorted["ARI"],
    color=[colors[alg] for alg in ari_sorted.index]
)

plt.title("Average ARI per Algorithm (Best → Worst)")
plt.ylabel("ARI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
mod_sorted = avg_alg.sort_values(by="Modularity", ascending=False)

plt.figure(figsize=(10,5))

plt.bar(
    mod_sorted.index,
    mod_sorted["Modularity"],
    color=[colors[alg] for alg in mod_sorted.index]
)

plt.title("Average Modularity per Algorithm (Best → Worst)")
plt.ylabel("Modularity")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
avg_alg["Score"] = (
    0.4 * avg_alg["NMI"] +
    0.4 * avg_alg["ARI"] +
    0.2 * avg_alg["Modularity"]
)

ranking = avg_alg["Score"].sort_values(ascending=False)

best = ranking.index[0]

plt.figure(figsize=(10,6))

plt.barh(
    ranking.index,
    ranking.values,
    color=[colors[alg] for alg in ranking.index]
)

# highlight best
plt.barh(
    ranking.index,
    ranking.values,
    color=["gold" if alg == best else colors[alg] for alg in ranking.index]
)

plt.title("Final Algorithm Ranking (Best → Worst)")
plt.xlabel("Weighted Score")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()